# 03 — E1 ViBidLQA MRC Fine-tuning
**Done Team | Ngày 2–3 | Pilot trước, full sau**

Mục tiêu: so sánh checkpoint XLM-RoBERTa trước/sau fine-tuning trên cùng câu hỏi, context và metric. Notebook 01/02 được giữ nguyên. Không dùng dữ liệu VNAB trong E1. Không đọc test để phát triển mô hình. Mặc định PILOT: 100 train, đúng 20 dev từ E1, 1 epoch. FULL là lượt chạy mới từ checkpoint gốc, không tiếp tục từ pilot; 1.928 train, 482 dev, tối đa 3 epoch. Kết quả dev không phải kết quả test độc lập. Chỉ chạy mục 10 sau khi khóa cấu hình chính thức.

**Cách chạy:** Colab GPU T4, chạy từng mục. Nếu phải cài thư viện, restart session và chạy lại 01. Mỗi mục chỉ có một cell code. Không sửa file 01/02. Không đưa token hoặc checkpoint nặng vào Git.

**Trạng thái:** notebook đã được tạo và kiểm tra cú pháp, chưa chạy thực nghiệm GPU. Các số liệu chỉ được sinh khi bạn thực thi.

## 01. Kiểm tra thư viện và GPU
Dùng phiên bản đã chạy ở E1/E2. Không cài lại PyTorch. CPU không phải lựa chọn mặc định để fine-tune toàn bộ 277 triệu tham số. Nếu cần cài thư viện, cell dừng có chủ đích; restart rồi chạy lại. Cảnh báo xung đột Gradio/Diffusers có thể xuất hiện như E1.

In [1]:
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
REQUIRED = {"transformers":"4.57.1", "datasets":"4.8.5", "huggingface-hub":"0.36.2", "accelerate":"1.15.0", "sentencepiece":"0.2.2"}
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None
changes = [f"{k}=={v}" for k,v in REQUIRED.items() if installed(k)!=v]
changes += [k for k in ("pandas","protobuf","tqdm") if installed(k) is None]
if changes:
    print("Cần cài:", changes, flush=True)
    subprocess.check_call([sys.executable,"-m","pip","install",*changes])
    raise SystemExit("Cài xong. Restart session rồi chạy lại mục 01.")
import torch, transformers, datasets, pandas as pd, numpy as np
import json, re, unicodedata, time, math, shutil, random, hashlib, platform
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline, set_seed
from torch.utils.data import DataLoader
from IPython.display import display
SEED = 42
set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại 01. Không cài lại torch.")
DEVICE = torch.device("cuda:0")
print("GPU:",torch.cuda.get_device_name(0), "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
print("CUDA check:",(torch.ones(1,device=DEVICE)*2).item())
for k in REQUIRED: print(k,installed(k))
print("torch",torch.__version__)
print("ENVIRONMENT OK")

GPU: Tesla T4 VRAM GiB: 14.56
CUDA check: 2.0
transformers 4.57.1
datasets 4.8.5
huggingface-hub 0.36.2
accelerate 1.15.0
sentencepiece 0.2.2
torch 2.14.1+cu130
ENVIRONMENT OK


## 02. Cấu hình và thư mục kết quả
PILOT chỉ kiểm tra học được và lưu checkpoint. F1 dev là tiêu chí chọn checkpoint, EM phân xử nếu bằng; nếu vẫn bằng, giữ epoch sớm hơn. FULL sẽ chạy lại từ checkpoint gốc. Để tránh ghi đè, mỗi lượt chạy tạo thư mục riêng. `gradient_accumulation` gom nhiều microbatch trước một optimizer step. FP16 giảm VRAM trên T4. Không thay đổi metric giữa hai cấu hình.

In [2]:
MODE = "FULL"  # Ngày 2: PILOT. Ngày 3: FULL (restart rồi chạy lại từ 01)
assert MODE in ("PILOT","FULL")
DATASET_NAME = "ntphuc149/ViBidLQA_v1"
DATASET_REVISION = "dc439e1344ec6a55a36027f1868b36d93eec840d"
MODEL_NAME = "deepset/xlm-roberta-base-squad2"
MODEL_REVISION = "a5fab9908c8d856e8c583fd41ba6d92444e46477"
TRAIN_SIZE = 100 if MODE == "PILOT" else 1928
EPOCHS = 1 if MODE == "PILOT" else 3
BATCH_SIZE = 2
ACCUM_STEPS = 4
LR = 2e-5
WEIGHT_DECAY = 0.01
MAX_LENGTH = 384
STRIDE = 128
USE_FP16 = True
QA_SETTINGS = {"max_seq_len":MAX_LENGTH,"doc_stride":STRIDE,"max_answer_len":128,"handle_impossible_answer":False}
PILOT_IDS = ['validation_00479', 'validation_00294', 'validation_00131', 'validation_00375', 'validation_00471', 'validation_00040', 'validation_00423', 'validation_00152', 'validation_00268', 'validation_00180', 'validation_00328', 'validation_00438', 'validation_00140', 'validation_00292', 'validation_00185', 'validation_00183', 'validation_00176', 'validation_00230', 'validation_00226', 'validation_00051']
RUN_DIR = Path("/content/E1_runs") / (MODE.lower()+"_"+time.strftime("%Y%m%d_%H%M%S"))
RUN_DIR.mkdir(parents=True,exist_ok=False)
BEST_DIR = RUN_DIR / "best_checkpoint"
config = {"mode":MODE,"dataset":DATASET_NAME,"dataset_revision":DATASET_REVISION,"model":MODEL_NAME,"model_revision":MODEL_REVISION,"seed":SEED,"train_size":TRAIN_SIZE,"epochs":EPOCHS,"batch_size":BATCH_SIZE,"gradient_accumulation":ACCUM_STEPS,"learning_rate":LR,"weight_decay":WEIGHT_DECAY,"max_length":MAX_LENGTH,"stride":STRIDE,"fp16":USE_FP16,"optimizer":"AdamW, constant LR, clip grad norm 1.0","qa_settings":QA_SETTINGS,"selection":"dev F1, then EM, then earliest epoch","metric_normalization":"NFC, lowercase, whitespace; retain punctuation and Vietnamese accents","training_windows":"positive windows only; select best enclosing candidate per window; prefer exact NFC-trim boundaries; no-answer windows omitted","device":str(DEVICE),"gpu":torch.cuda.get_device_name(0),"python":platform.python_version(),"libraries":{**{k:installed(k) for k in REQUIRED},"torch":str(torch.__version__)}}
def save_json(name,obj):
    (RUN_DIR/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding="utf-8")
config["grad_scaler_init_scale"] = 1024.0
save_json("config.json", config)
print(json.dumps(config,ensure_ascii=False,indent=2))

{
  "mode": "FULL",
  "dataset": "ntphuc149/ViBidLQA_v1",
  "dataset_revision": "dc439e1344ec6a55a36027f1868b36d93eec840d",
  "model": "deepset/xlm-roberta-base-squad2",
  "model_revision": "a5fab9908c8d856e8c583fd41ba6d92444e46477",
  "seed": 42,
  "train_size": 1928,
  "epochs": 3,
  "batch_size": 2,
  "gradient_accumulation": 4,
  "learning_rate": 2e-05,
  "weight_decay": 0.01,
  "max_length": 384,
  "stride": 128,
  "fp16": true,
  "optimizer": "AdamW, constant LR, clip grad norm 1.0",
  "qa_settings": {
    "max_seq_len": 384,
    "doc_stride": 128,
    "max_answer_len": 128,
    "handle_impossible_answer": false
  },
  "selection": "dev F1, then EM, then earliest epoch",
  "metric_normalization": "NFC, lowercase, whitespace; retain punctuation and Vietnamese accents",
  "training_windows": "positive windows only; select best enclosing candidate per window; prefer exact NFC-trim boundaries; no-answer windows omitted",
  "device": "cuda:0",
  "gpu": "Tesla T4",
  "python": "3.13.16

## 03. Tải train/dev, chuẩn hóa NFC và chọn mẫu
Giữ nguyên split công bố. Tải riêng train và validation, không đọc test. Tất cả vị trí đáp án trùng chuỗi là ứng viên. PILOT lấy 100 train ngẫu nhiên cố định và chính xác 20 ID validation đã lưu trong ZIP E1; FULL dùng toàn bộ train/dev. Xuất manifest để đối chiếu.

In [3]:
def nfc(s): return unicodedata.normalize("NFC",s)
def prepare_split(raw,split):
    records=[]
    for i,row in enumerate(raw):
        ctx,q,ans = [nfc(row[k] or "") for k in ("context","question","answer")]
        assert all(x.strip() for x in (ctx,q,ans)), (split,i,"empty")
        positions=[m.start() for m in re.finditer(re.escape(ans),ctx)]
        assert positions, (split,i,"no span")
        assert all(ctx[p:p+len(ans)]==ans for p in positions)
        records.append({"id":f"{split}_{i:05d}","context":ctx,"question":q,"answers":{"text":[ans]*len(positions),"answer_start":positions}})
    return Dataset.from_list(records)
train_raw = load_dataset(DATASET_NAME,revision=DATASET_REVISION,split="train")
dev_raw = load_dataset(DATASET_NAME,revision=DATASET_REVISION,split="validation")
assert (len(train_raw),len(dev_raw))==(1928,482)
train_all=prepare_split(train_raw,"train")
dev_all=prepare_split(dev_raw,"validation")
train_data=train_all.shuffle(seed=SEED).select(range(TRAIN_SIZE)) if MODE=="PILOT" else train_all
if MODE=="PILOT":
    lookup={r["id"]:i for i,r in enumerate(dev_all)}
    dev_data=dev_all.select([lookup[x] for x in PILOT_IDS])
    assert list(dev_data["id"])==PILOT_IDS
else: dev_data=dev_all
save_json("split_ids.json",{"train":list(train_data["id"]),"validation":list(dev_data["id"])})
print("Train:",len(train_data),"Dev:",len(dev_data),"Test not loaded")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train: 1928 Dev: 482 Test not loaded


## 04. Nạp mô hình và tokenizer
QA head dự đoán start/end logits. Checkpoint gốc được giữ nguyên để tính baseline trước khi huấn luyện. Chỉ có một model trong VRAM tại một thời điểm.

In [4]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,use_fast=True)
model,info=AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,output_loading_info=True)
assert tokenizer.is_fast and tokenizer.padding_side=="right"
assert not info["missing_keys"] and not info["mismatched_keys"],info
model.to(DEVICE)
model.eval()
config.update({"tokenizer":type(tokenizer).__name__,"parameters":sum(p.numel() for p in model.parameters())})
save_json("config.json",config)
print("Model:",MODEL_NAME,"Fast tokenizer:",tokenizer.is_fast,"Parameters:",config["parameters"])
print("MODEL LOADED")

Model: deepset/xlm-roberta-base-squad2 Fast tokenizer: True Parameters: 277454594
MODEL LOADED


## 05. Tokenization cho huấn luyện — sliding window
Context dài được chia cửa sổ 384 token, overlap 128 token. Với mỗi cửa sổ có ít nhất một gold span trọn vẹn, chọn một ứng viên khớp tốt nhất, ưu tiên ranh giới đúng sau NFC/trim; nếu ranh giới subword không khớp hoàn toàn, giữ span bao phủ gần nhất và ghi audit. Các cửa sổ không có đáp án bị bỏ qua (không phải mẫu unanswerable). Không bỏ âm thầm câu hỏi không có cửa sổ hợp lệ: dừng và xem audit. Đếm số câu hỏi và số feature riêng biệt; một câu có thể sinh nhiều feature.

In [5]:
def make_features(row):
    ctx,q=row["context"],row["question"].strip()
    encoded=tokenizer(q,ctx,truncation="only_second",max_length=MAX_LENGTH,stride=STRIDE,return_overflowing_tokens=True,return_offsets_mapping=True,padding=False)
    features,audit=[],[]
    for j,offsets in enumerate(encoded["offset_mapping"]):
        seq=encoded.sequence_ids(j)
        valid=[i for i,(a,b) in enumerate(offsets) if seq[i]==1 and b>a]
        candidates=[]
        for text,start in zip(row["answers"]["text"],row["answers"]["answer_start"]):
            end=start+len(text)
            left=[i for i in valid if offsets[i][0]<=start<offsets[i][1]]
            right=[i for i in valid if offsets[i][0]<end<=offsets[i][1]]
            for si in left:
                for ei in right:
                    if si>ei: continue
                    a,b=offsets[si][0],offsets[ei][1]
                    if not (a<=start<end<=b): continue
                    extracted=ctx[a:b]
                    match=extracted.strip()==text.strip()
                    candidates.append((not match,(start-a)+(b-end),si,ei,extracted,text,start,end))
        if not candidates: continue
        mismatch,overhang,si,ei,extracted,text,start,end=min(candidates)
        feature={k:encoded[k][j] for k in tokenizer.model_input_names if k in encoded}
        feature.update(start_positions=si,end_positions=ei)
        features.append(feature)
        audit.append({"id":row["id"],"window":j,"start_token":si,"end_token":ei,"gold":text,"token_span":extracted,"boundary_match":not mismatch,"overhang_chars":overhang,"answer_tokens":ei-si+1})
    return features,audit,len(encoded["input_ids"])
train_features=[]; audits=[]; coverage=[]
for row in tqdm(train_data,desc="Tokenize train"):
    feats,logs,n_windows=make_features(row)
    train_features.extend(feats); audits.extend(logs)
    coverage.append({"id":row["id"],"windows":n_windows,"positive_windows":len(feats)})
coverage_df=pd.DataFrame(coverage); audit_df=pd.DataFrame(audits)
coverage_df.to_csv(RUN_DIR/"train_coverage.csv",index=False)
audit_df.to_csv(RUN_DIR/"token_audit.csv",index=False,encoding="utf-8-sig")
missing=coverage_df.loc[coverage_df.positive_windows==0,"id"].tolist()
print("Train examples:",len(train_data),"Positive windows:",len(train_features),"No positive window:",len(missing))
assert not missing, f"Cần rà mẫu không có span token: {missing}"
print("Boundary mismatches:",int((~audit_df.boundary_match).sum()))
print("Positive windows with answer > inference max_answer_len:",int((audit_df.answer_tokens>QA_SETTINGS["max_answer_len"]).sum()))
print("TOKENIZATION OK")
# Dynamic padding: chỉ đệm tới chuỗi dài nhất trong batch.
def collate(rows):
    batch=tokenizer.pad([{k:v for k,v in r.items() if k not in ("start_positions","end_positions")} for r in rows],padding=True,return_tensors="pt")
    for k in ("start_positions","end_positions"): batch[k]=torch.tensor([r[k] for r in rows],dtype=torch.long)
    return batch

Tokenize train:   0%|          | 0/1928 [00:00<?, ?it/s]

Train examples: 1928 Positive windows: 2030 No positive window: 0
Boundary mismatches: 74
Positive windows with answer > inference max_answer_len: 0
TOKENIZATION OK


## 06. Baseline trước fine-tuning trên cùng dev
Giữ chính xác cách chấm notebook 01: NFC + lowercase + whitespace, giữ dấu câu và dấu tiếng Việt. EM là khớp hoàn toàn, F1 là token overlap. Không so độ trễ CPU/GPU trực tiếp. Inference nhận context cho trước và không dùng gold để chọn đáp án.

In [6]:
def normalize_answer(s): return " ".join(nfc(s).lower().split())
def score_one(pred,gold):
    p,g=normalize_answer(pred),normalize_answer(gold)
    pt,gt=p.split(),g.split()
    em=int(p==g)
    if not pt or not gt: return em,float(pt==gt)
    common=sum((Counter(pt)&Counter(gt)).values())
    return em,2*common/(len(pt)+len(gt))
assert score_one("Sài Gòn","phường Sài Gòn")== (0,0.8)
assert score_one("phường Sài Gòn.","phường Sài Gòn")[0]==0

def evaluate(reader, data, tag):
    reader.eval()
    qa=pipeline("question-answering",model=reader,tokenizer=tokenizer,device=0)
    records=[]
    for row in tqdm(data,desc=tag):
        torch.cuda.synchronize(); tic=time.perf_counter()
        result=qa(question=row["question"],context=row["context"],**QA_SETTINGS)
        torch.cuda.synchronize(); ms=(time.perf_counter()-tic)*1000
        pred=result["answer"]; gold=row["answers"]["text"][0]
        em,f1=score_one(pred,gold)
        records.append({"id":row["id"],"question":row["question"],"gold":gold,"prediction":pred,"score":float(result["score"]),"pred_start":result["start"],"pred_end":result["end"],"em":em,"f1":f1,"latency_ms":ms})
    df=pd.DataFrame(records)
    metrics={"tag":tag,"samples":len(df),"EM_percent":float(df.em.mean()*100),"F1_percent":float(df.f1.mean()*100),"avg_latency_ms":float(df.latency_ms.mean())}
    df.to_csv(RUN_DIR/f"{tag}_predictions.csv",index=False,encoding="utf-8-sig")
    save_json(f"{tag}_metrics.json",metrics)
    del qa
    return metrics
baseline=evaluate(model,dev_data,"baseline_dev")
print(json.dumps(baseline,indent=2))

Device set to use cuda:0


baseline_dev:   0%|          | 0/482 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{
  "tag": "baseline_dev",
  "samples": 482,
  "EM_percent": 10.995850622406639,
  "F1_percent": 56.90418661329477,
  "avg_latency_ms": 24.17239302073721
}


## 07. Fine-tuning — forward, loss, backward, optimizer
Đây là phần mới của Ngày 2. `model.train()` bật chế độ huấn luyện; loss so start/end logits với nhãn. `backward()` tính gradient; AdamW cập nhật trọng số. Dùng mixed precision FP16 + GradScaler và gradient accumulation. Lưu loss từng optimizer step, kiểm tra có ít nhất một bước cập nhật thực sự và một trọng số QA head đã đổi. Sau mỗi epoch tính EM/F1 dev, lưu checkpoint tốt nhất theo F1 rồi EM. Không chạy lại riêng cell này nếu đã train; restart và tạo lượt chạy mới để giữ baseline sạch. Nếu OOM: restart, giảm BATCH_SIZE tại mục 02 xuống 1 (tăng ACCUM_STEPS nếu muốn giữ effective batch), chạy lại từ đầu và ghi rõ thay đổi.

In [7]:
assert not BEST_DIR.exists(), "Đã có checkpoint lượt này. Restart và chạy từ đầu để train lại."
set_seed(SEED)
generator=torch.Generator().manual_seed(SEED)
loader=DataLoader(train_features,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate,generator=generator,num_workers=0)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_FP16,
    init_scale=config["grad_scaler_init_scale"]
)
initial_head=model.qa_outputs.weight.detach().cpu().clone()
training_log=[]; history=[]; best_key=None; best_epoch=None; updates=0
for epoch in range(1,EPOCHS+1):
    model.train(); optimizer.zero_grad(set_to_none=True)
    group_loss=0.0; group_examples=0; group_batches=0
    # Gom microbatch, chuẩn hóa gradient theo số mẫu trong nhóm (kể cả nhóm cuối).
    for i,batch in enumerate(tqdm(loader,desc=f"Train epoch {epoch}"),start=1):
        batch={k:v.to(DEVICE) for k,v in batch.items()}
        n=batch["input_ids"].shape[0]
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=USE_FP16):
            out=model(**batch); loss=out.loss
        raw=float(loss.detach().item())
        if not math.isfinite(raw): raise RuntimeError(f"Non-finite loss: epoch={epoch}, batch={i}")
        scaler.scale(loss*n).backward()
        group_loss+=raw*n; group_examples+=n; group_batches+=1
        del out,loss,batch
        if i%ACCUM_STEPS==0 or i==len(loader):
            scaler.unscale_(optimizer)
            for p in model.parameters():
                if p.grad is not None: p.grad.div_(group_examples)
            grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scale_before=scaler.get_scale(); scaler.step(optimizer); scaler.update()
            skipped=scaler.get_scale()<scale_before
            if not skipped: updates+=1
            training_log.append({"epoch":epoch,"batch_end":i,"optimizer_updates":updates,"loss":group_loss/group_examples,"gradient_norm":float(grad_norm) if torch.isfinite(grad_norm) else None,"skipped":bool(skipped),"features":group_examples})
            optimizer.zero_grad(set_to_none=True)
            group_loss=0.0;group_examples=0;group_batches=0
    pd.DataFrame(training_log).to_csv(RUN_DIR/"training_log.csv",index=False)
    assert updates>0, "Chưa có optimizer step thành công"
    met=evaluate(model,dev_data,f"epoch_{epoch}_dev")
    history.append({"epoch":epoch,**met})
    save_json("dev_history.json",history)
    key=(met["F1_percent"],met["EM_percent"])
    if best_key is None or key>best_key:
        best_key=key;best_epoch=epoch
        model.save_pretrained(BEST_DIR,safe_serialization=True)
        tokenizer.save_pretrained(BEST_DIR)
        save_json("best_selection.json",{"epoch":epoch,"metrics":met,"criterion":config["selection"]})
    print("Epoch",epoch,"dev:",met,"best epoch:",best_epoch)
head_delta=float((model.qa_outputs.weight.detach().cpu()-initial_head).abs().max())
assert head_delta>0 and updates>0
save_json("training_status.json",{"optimizer_updates":updates,"head_max_abs_change_final":head_delta,"best_epoch":best_epoch,"epochs_completed":EPOCHS})
print("TRAINING OK — optimizer updates:",updates,"QA head max abs change:",head_delta)

Train epoch 1:   0%|          | 0/1015 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Device set to use cuda:0


epoch_1_dev:   0%|          | 0/482 [00:00<?, ?it/s]

Epoch 1 dev: {'tag': 'epoch_1_dev', 'samples': 482, 'EM_percent': 22.61410788381743, 'F1_percent': 69.00018895142634, 'avg_latency_ms': 20.959428890039955} best epoch: 1


Train epoch 2:   0%|          | 0/1015 [00:00<?, ?it/s]

Device set to use cuda:0


... (2 outputs left out)

Train epoch 3:   0%|          | 0/1015 [00:00<?, ?it/s]

Device set to use cuda:0


epoch_3_dev:   0%|          | 0/482 [00:00<?, ?it/s]

Epoch 3 dev: {'tag': 'epoch_3_dev', 'samples': 482, 'EM_percent': 24.688796680497926, 'F1_percent': 69.859259234075, 'avg_latency_ms': 20.972282165974857} best epoch: 2
TRAINING OK — optimizer updates: 760 QA head max abs change: 0.00238836370408535


## 08. So sánh và xác nhận checkpoint tốt nhất
Giải phóng model/optimizer cũ rồi nạp lại checkpoint đã chọn trên dev. So sánh lại trên đúng dev IDs. Kết quả âm hoặc không cải thiện vẫn hợp lệ. PILOT không được gọi là kết quả chính thức.

In [8]:
import gc
assert BEST_DIR.exists() and best_epoch is not None
# Không giữ hai model cùng lúc trong GPU.
del optimizer,scaler,model
# vòng for có thể giữ p cuối cùng; xóa tham chiếu nếu có
if "p" in globals(): del p
gc.collect(); torch.cuda.empty_cache()
model=AutoModelForQuestionAnswering.from_pretrained(BEST_DIR).to(DEVICE)
model.eval()
best_dev=evaluate(model,dev_data,"best_checkpoint_dev")
comparison=pd.DataFrame([{"configuration":"before_finetuning",**baseline},{"configuration":"after_finetuning",**best_dev}])
comparison.to_csv(RUN_DIR/"dev_comparison.csv",index=False)
display(comparison)
print("Delta F1 (percentage points):",best_dev["F1_percent"]-baseline["F1_percent"])
print("Delta EM (percentage points):",best_dev["EM_percent"]-baseline["EM_percent"])
print("Best epoch:",best_epoch)

Device set to use cuda:0


best_checkpoint_dev:   0%|          | 0/482 [00:00<?, ?it/s]

,configuration,tag,samples,EM_percent,F1_percent,avg_latency_ms
0,before_finetuning,baseline_dev,482,10.995851,56.904187,24.172393
1,after_finetuning,best_checkpoint_dev,482,21.991701,69.926734,21.165758


Delta F1 (percentage points): 13.022547174965823
Delta EM (percentage points): 10.995850622406639
Best epoch: 2


## 09. Lưu log và checkpoint — bắt buộc trước khi ngắt Colab
ZIP nhỏ chỉ chứa báo cáo, không chứa checkpoint. Checkpoint khoảng 1.1 GB: lưu riêng vào Google Drive hoặc tải ZIP checkpoint riêng. Không đưa file trọng số lên Git. Nếu không lưu checkpoint ra ngoài /content, phiên Colab kết thúc có thể mất trọng số. `SAVE_TO_DRIVE=True` sẽ yêu cầu cấp quyền Drive; nếu không muốn, dùng False rồi tải cả hai ZIP. Chỉ PILOT: dừng tại đây. FULL: khóa cấu hình dựa trên dev, sau đó có thể chạy 10 để test E1.

In [9]:
from google.colab import files, drive
SAVE_TO_DRIVE = True
assert (BEST_DIR/"model.safetensors").is_file()
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()
checkpoint_hashes={str(p.relative_to(BEST_DIR)):sha256_file(p) for p in sorted(BEST_DIR.rglob("*")) if p.is_file()}
save_json("checkpoint_sha256.json",checkpoint_hashes)
# Ghi lại mọi quyết định trước test; pilot không được coi là khóa chính thức.
save_json("selection_manifest.json",{"mode":MODE,"test_allowed":MODE=="FULL","best_epoch":best_epoch,"config":config,"checkpoint_sha256":checkpoint_hashes,"dev_metrics":best_dev})
report_dir=RUN_DIR/"report_export";report_dir.mkdir(exist_ok=True)
for p in RUN_DIR.iterdir():
    if p.is_file() and p.suffix in (".json",".csv"): shutil.copy2(p,report_dir/p.name)
report_zip=shutil.make_archive(str(RUN_DIR)+"_results","zip",root_dir=report_dir)
if SAVE_TO_DRIVE:
    drive.mount("/content/drive")
    target=Path("/content/drive/MyDrive/VNAB_research/E1_runs")/RUN_DIR.name
    target.parent.mkdir(parents=True,exist_ok=True)
    shutil.copytree(RUN_DIR,target,dirs_exist_ok=True)
    assert sha256_file(target/"best_checkpoint/model.safetensors")==checkpoint_hashes["model.safetensors"]
    print("Saved checkpoint and logs to Drive:",target)
else:
    checkpoint_zip=shutil.make_archive(str(RUN_DIR)+"_checkpoint","zip",root_dir=RUN_DIR,base_dir="best_checkpoint")
    files.download(checkpoint_zip)
files.download(report_zip)
print("RESULTS SAVED:",report_zip)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved checkpoint and logs to Drive: /content/drive/MyDrive/VNAB_research/E1_runs/full_20261127_143558


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RESULTS SAVED: /content/E1_runs/full_20261127_143558_results.zip


## 10. Tùy chọn — E1 test chính thức (chỉ FULL, sau khi khóa)
**Không chạy mục này hôm nay khi MODE=PILOT.** Chỉ đặt `CONFIRM_LOCKED=True` sau khi nhóm xác nhận phương pháp, metric, config và checkpoint đã khóa, và đã lưu checkpoint ra ngoài runtime. Tải split test công bố lần đầu tại đây. Nếu có câu không có span sau NFC, dừng và báo lỗi thay vì tự ý lọc; không thay đổi mô hình dựa vào test. So sánh checkpoint gốc và checkpoint đã khóa cùng 603 câu, context và metric. Đây là test E1 công khai, không phải test VNAB của Vi.

In [10]:
CONFIRM_LOCKED=True
if MODE!="FULL" or not CONFIRM_LOCKED:
    print("SKIP: Chưa khóa FULL; không tải test.")
else:
    manifest=json.loads((RUN_DIR/"selection_manifest.json").read_text())
    assert manifest["test_allowed"] and manifest["config"]==config
    assert all(sha256_file(BEST_DIR/k)==v for k,v in manifest["checkpoint_sha256"].items())
    test_raw=load_dataset(DATASET_NAME,revision=DATASET_REVISION,split="test")
    assert len(test_raw)==603
    test_data=prepare_split(test_raw,"test")
    save_json("test_ids.json",list(test_data["id"]))
    del model; gc.collect();torch.cuda.empty_cache()
    model=AutoModelForQuestionAnswering.from_pretrained(BEST_DIR).to(DEVICE)
    test_after=evaluate(model,test_data,"after_test")
    del model; gc.collect();torch.cuda.empty_cache()
    model=AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME,revision=MODEL_REVISION).to(DEVICE)
    test_before=evaluate(model,test_data,"before_test")
    test_comparison=pd.DataFrame([test_before,test_after]);display(test_comparison)
    test_comparison.to_csv(RUN_DIR/"test_comparison.csv",index=False)
    for p in RUN_DIR.iterdir():
        if p.is_file() and p.suffix in (".json",".csv"): shutil.copy2(p,report_dir/p.name)
    archive=shutil.make_archive(str(RUN_DIR)+"_official_results","zip",root_dir=report_dir)
    if SAVE_TO_DRIVE: shutil.copytree(RUN_DIR,target,dirs_exist_ok=True)
    files.download(archive)
    print("Official E1 results saved; do not tune on test.")

Device set to use cuda:0


after_test:   0%|          | 0/603 [00:00<?, ?it/s]

Device set to use cuda:0


before_test:   0%|          | 0/603 [00:00<?, ?it/s]

,tag,samples,EM_percent,F1_percent,avg_latency_ms
0,before_test,603,15.422886,59.496952,40.959731
1,after_test,603,30.016584,71.852935,37.644798


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Official E1 results saved; do not tune on test.
